# 04 · External validation on TCGA, by race, sex and age

**Why.** Our models are trained on UCSF-PDGM: one hospital, and its public release has no race field. To check whether the models hold up on other hospitals and across racial groups, we run the **deployed models, unchanged**, on TCGA-GBM/LGG patients from many US hospitals.

- **Images and tumor outlines:** BraTS 2020 training set (pre-operative TCGA scans; same four sequences and labels as UCSF-PDGM).
- **Lab results:** cBioPortal `lgggbm_tcga_pub` (IDH, 1p/19q, MGMT, grade, age, sex).
- **Race:** NCI Genomic Data Commons demographic records.

Features were extracted with `ml/pipeline/extract.py` (mode v1, identical to training) by `ml/pipeline/external_validation.py`. Only the derived feature table is committed; this notebook reruns every number from it, without the raw scans.

Race is **evaluated, not used as a model input**: it is a social category, not a measurement, and as an input it could encode existing bias.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd

REPO=Path.cwd().resolve().parents[1]
sys.path[:0]=[str(REPO/"backend"), str(REPO/"ml"/"pipeline")]
import external_validation as ev
from app.core.logging import configure_logging
configure_logging("WARNING")

table=pd.read_parquet(ev.FEATURES)
print(len(table), "patients with scans and lab results")
table["race"].value_counts()

167

patients with scans and lab results

race
White                        147
Black or African American     12
Other or not reported          5
Asian                          3
Name: count, dtype: int64

## Who is in the external cohort

In [2]:
pd.DataFrame({
    "patients": [len(table)],
    "female": [int((table["sex"]==0).sum())], "male": [int((table["sex"]==1).sum())],
    "median age": [table["age"].median()],
    "IDH-mutant": [int((table["idh"]==1).sum())], "1p/19q codeleted": [int((table["codeletion_1p19q"]==1).sum())],
    "grade 4": [int((table["who_grade"]==4).sum())],
})

,patients,female,male,median age,IDH-mutant,1p/19q codeleted,grade 4
0,167,76,90,54.0,57,13,101


## Run the deployed models (nothing refitted)

In [3]:
preds=ev.predict(table)
result=ev.report(table, preds)
overall=pd.DataFrame({m: e["overall"] for m, e in result["markers"].items()}).T
overall

,n,positives,accuracy,auc,balanced_accuracy,stable_n,stable_accuracy
1p19q,143.0,13.0,0.9091,0.3923,0.5000,143.0,0.9091
GRADE,166.0,101.0,0.6566,0.9021,0.5615,137.0,0.7299
IDH,147.0,57.0,0.7347,0.7889,0.7094,85.0,0.8000
MGMT,127.0,79.0,0.6220,0.5883,0.5000,NaN,NaN


## Performance by race

AUC: how well patients are ranked (0.5 = chance). Accuracy: calls at the model's cut-off. Groups under 10 patients are too small to compare.

In [4]:
def by(group):
    rows=[]
    for m, e in result["markers"].items():
        for g, x in e[f"by_{group}"].items():
            rows.append({"marker": m, group: g, **x})
    return pd.DataFrame(rows).set_index(["marker", group])
by("race")

n  positives  accuracy  stable_n  \
marker race                                                            
1p19q  Asian                        2          0    1.0000       2.0   
       Black or African American   11          0    1.0000      11.0   
       Other or not reported        3          0    1.0000       3.0   
       White                      127         13    0.8976     127.0   
GRADE  Asian                        3          3    1.0000       3.0   
       Black or African American   12          7    0.5833      10.0   
       Other or not reported        4          4    1.0000       4.0   
       White                      147         87    0.6463     120.0   
IDH    Asian                        2          0    0.5000       2.0   
       Black or African American   11          5    0.6364       5.0   
       Other or not reported        4          0    0.7500       3.0   
       White                      130         52    0.7462      75.0   
MGMT   Asian                        1          1    1.0000       NaN   
       Black or African American   10          8    0.8000       NaN   
       Other or not reported        2          0    0.0000       NaN   
       White                      114         70    0.6140       NaN   

                                  stable_accuracy     auc  balanced_accuracy  
marker race                                                                   
1p19q  Asian                               1.0000     NaN                NaN  
       Black or African American           1.0000     NaN                NaN  
       Other or not reported               1.0000     NaN                NaN  
       White                               0.8976  0.3914             0.5000  
GRADE  Asian                               1.0000     NaN                NaN  
       Black or African American           0.7000  0.9429             0.5000  
       Other or not reported               1.0000     NaN                NaN  
       White                               0.7167  0.9033             0.5667  
IDH    Asian                               0.5000     NaN                NaN  
       Black or African American           0.6000  0.8667             0.6167  
       Other or not reported               1.0000     NaN                NaN  
       White                               0.8133  0.7941             0.7244  
MGMT   Asian                                  NaN     NaN                NaN  
       Black or African American              NaN  0.8125             0.5000  
       Other or not reported                  NaN     NaN                NaN  
       White                                  NaN  0.5838             0.5000

## By sex and age

In [5]:
by("sex")

n  positives  accuracy     auc  balanced_accuracy  stable_n  \
marker sex                                                                    
1p19q  female  69          9    0.8696  0.3426             0.5000      69.0   
       male    74          4    0.9459  0.4929             0.5000      74.0   
GRADE  female  76         38    0.5658  0.8885             0.5658      58.0   
       male    90         63    0.7333  0.9101             0.5556      79.0   
IDH    female  71         34    0.6901  0.7599             0.6860      37.0   
       male    76         23    0.7763  0.8171             0.7289      48.0   
MGMT   female  68         45    0.6618  0.6280             0.5000       NaN   
       male    59         34    0.5763  0.5541             0.5000       NaN   

               stable_accuracy  
marker sex                      
1p19q  female           0.8696  
       male             0.9459  
GRADE  female           0.6379  
       male             0.7975  
IDH    female           0.7297  
       male             0.8542  
MGMT   female              NaN  
       male                NaN

In [6]:
by("age_band")

n  positives  accuracy     auc  balanced_accuracy  stable_n  \
marker age_band                                                                 
1p19q  45-64     67         11    0.8358  0.3896             0.5000      67.0   
       65+       27          1    0.9630  0.0000             0.5000      27.0   
       under 45  49          1    0.9796  0.9583             0.5000      49.0   
GRADE  45-64     82         56    0.7195  0.8970             0.5577      71.0   
       65+       34         30    0.8824  0.9667             0.5000      33.0   
       under 45  50         15    0.4000  0.8419             0.5714      33.0   
IDH    45-64     69         20    0.7101  0.6796             0.6332      40.0   
       65+       29          3    0.8621  0.6667             0.4808      20.0   
       under 45  49         34    0.6939  0.7294             0.6676      25.0   
MGMT   45-64     60         35    0.5833  0.6377             0.5000       NaN   
       65+       22         13    0.5909  0.3504             0.5000       NaN   
       under 45  45         31    0.6889  0.6406             0.5000       NaN   

                 stable_accuracy  
marker age_band                   
1p19q  45-64              0.8358  
       65+                0.9630  
       under 45           0.9796  
GRADE  45-64              0.7887  
       65+                0.9091  
       under 45           0.4242  
IDH    45-64              0.8000  
       65+                0.9500  
       under 45           0.6800  
MGMT   45-64                 NaN  
       65+                   NaN  
       under 45              NaN

## Do Stable calls stay more reliable on new hospitals?

In [7]:
pd.DataFrame([{"marker": m, "all calls accuracy": e["overall"].get("accuracy"),
               "stable calls": e["overall"].get("stable_n"), "stable accuracy": e["overall"].get("stable_accuracy")}
              for m, e in result["markers"].items()]).set_index("marker")

,all calls accuracy,stable calls,stable accuracy
marker,,,
1p19q,0.9091,143.0,0.9091
GRADE,0.6566,137.0,0.7299
IDH,0.7347,85.0,0.8000
MGMT,0.6220,NaN,NaN


## Matches the committed report used by the app

In [8]:
committed=json.loads(ev.METRICS.read_text())
assert committed["markers"]==json.loads(json.dumps(result["markers"])), "recomputed metrics differ"
print("recomputed metrics match artifacts/metrics/external_tcga.json")

recomputed metrics match artifacts/metrics/external_tcga.json

## What this shows

- Read the overall table against the UCSF held-out results in `artifacts/metrics/heldout.json` (model card): a drop on TCGA is the cost of moving to new scanners, which the stress test predicted for raw-intensity (v1) features.
- Black and Asian patients are few in TCGA (see counts above), so per-race differences here are not statistically meaningful. For Black patients (11-12 per marker) ranking (AUC) is similar to or higher than for White patients while accuracy at the cut-off is lower; with so few patients neither difference is statistically meaningful, and Asian patients (2-3) cannot be evaluated at all.
- Next step: retrain on brain-normalized (v2) features and add TCGA to training, then repeat this check.